<a href="https://colab.research.google.com/github/Klark-cyber/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Menu detector")

In [ ]:
# Google Colab muhitiga Google Drive-ni ulash uchun maxsus modulni yuklab olamiz
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms # imageni tensorga otkazish uchun transformsdan foydalanamiz
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader # Dataset classini import qildik
import os
import numpy as np


In [ ]:
# Google Drive-ni '/content/drive' papkasiga virtual disk sifatida ulaymiz
# Bu kod ishga tushganda Drive-ga kirish uchun ruxsat so'raydi
drive.mount('/content/drive')

In [ ]:
# Define Dataset Path -> Dataset joylashgan manzilni aniqlash
# Google Drive ichidagi 'food101_dataset' papkasining manzilini o'zgaruvchiga saqlaymiz
DATASET_PATH = '/content/drive/MyDrive/food101_dataset'

# Konsolga dataset manzilini tekshirish uchun chiqarib ko'rsatamiz
print('Dataset_path:', DATASET_PATH)

# Klasslarni o'zgartirish yoki qayta nomlash uchun xarita (mapping) yaratish
# Rasmda bu qism to'liq tugatilmagan, hozircha faqat 'hamburger' kiritilgan
CUSTOM_CLASS_MAPPING = {
    'hamburger': "hamburger",
    "hot_dog": "hot_dog",
    "chocolate_cake": "dessert",
    "cheesecake": "dessert",
    "kebab": "kebab",
    "pilaf": "pilaf"

}

# Model o'rganishi kerak bo'lgan taomlar klaslarining (nomlarining) ro'yxati
CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']

# Har bir klas nomiga mos ravishda tartib raqami (indeks) biriktiramiz: {'hamburger': 0, 'hot_dog': 1, ...}
# Buning uchun 'enumerate' funksiyasi va Dict Comprehension usulidan foydalanilgan
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}

# Jami nechta klas borligini aniqlaymiz (len funksiyasi ro'yxat elementlari sonini hisoblaydi)
NUM_CLASSES = len(CLASSES)

# Tuzilgan klaslar va ularning indekslari lug'atini (dictionary) konsolga chop etamiz
print(CLASS_TO_IDX)

transform = transforms.Compose([  # compose methodi bu simple ketma ketlikni amalga oshirishni qayt qilissh uchun kerak
    transforms.Resize((224, 224)), # Resuze methodi orqali barcha rasmni bir xil sizega keltirib olamiz
    transforms.ToTensor(), # bu method rasmlarni tensorga otkazib beradi,
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])   # barcha rasmlarni normal holatga keltiradi (scale+chanelni togri shaklga keltiradi)
])

# 0~255
#0.0~1.0
#0.234
#H, W, Channel => C,H,W | RGB = Red, Green, Blue
#Normalize
#pixel = (pixel - mean)/std



In [ ]:
# Custom Dataset Class
class FoodDataset(Dataset):  # PyTorch-ning Dataset klassidan voris olib, maxsus FoodDataset klassini yaratamiz
    def __init__(self, images, labels, transform=None):  # Klass initsializatori: rasmlar, teglar va transformatsiyalarni qabul qiladi
        self.images = images  # Kelgan rasmlar yo'li (path) ro'yxatini klass xususiyatiga biriktiramiz
        self.labels = labels  # Kelgan teglar (klaslar) ro'yxatini klass xususiyatiga biriktiramiz
        self.transform = transform  # Rasmlarga qo'llaniladigan augmentatsiya/transformatsiyalarni saqlaymiz

    def __len__(self):  # Dataset ichidagi ma'lumotlarning umumiy sonini qaytaruvchi maxsus metod
        print('images_length', len(self.images))  # Konsolga rasmlarning umumiy sonini chop etamiz
        return len(self.images)  # Dataset uzunligi sifatida rasmlar ro'yxati o'lchamini qaytaramiz

    def __getitem__(self, idx):  # Berilgan indeks (idx) bo'yicha bitta rasm va uning tegini qaytaruvchi metod
        img_path = self.images[idx]  # Berilgan indeksga mos keladigan rasm fayli yo'lini aniqlaymiz
        #print('image_path', img_path)  # Konsolga yuklanayotgan rasmning manzili (yo'li)ni chop etamiz
        label = self.labels[idx]  # Berilgan indeksga mos keladigan rasm tegini (label) aniqlaymiz
        #print('label', label)  # Konsolga rasm tegining qiymatini chop etamiz
        try:  # Rasmni ochishda yuzaga kelishi mumkin bo'lgan xatoliklarni tekshirish bloki
            image = Image.open(img_path)
            if image.mode == "P" or image.mode == "RGBA":  # Rasmni ochamiz va uni RGB rang formatiga o'tkazamiz
              image = image.convert('RGBA').convert("RGB")
            else:
              image = image.convert('RGB')
        except (UnidentifiedImageError, OSError):  # Agar rasm fayli buzilgan yoki ochib bo'lmaydigan bo'lsa xatolikni ushlaymiz
            print(f"Skipping broken image: {img_path}")  # Konsolga buzilgan rasm tashlab ketilgani haqida ogohlantirish chiqaramiz
            return self.getitem((idx + 1) % len(self.images))  # Keyingi indeksdagi rasmni qayta chaqirib, cheksiz sikldan qochish uchun qoldiq olamiz
        if self.transform:  # Agar rasm uchun transformatsiya obyekti mavjud bo'lsa
            image = self.transform(image)  # Rasmni belgilangan transformatsiyalardan (o'lcham, tensorga o'tkazish va h.k.) o'tkazamiz
        return image, label  # Tayyor bo'lgan rasm obyektini va unga mos keladigan tegni qaytaramiz

In [ ]:
# Gather and Split Data
# Ma'lumotlarni yig'ish va qismlarga ajratish uchun sarlavhali izohlar

all_images = []  # Barcha rasmlar yo'li va ularning teglarini juftlik (tuple) ko'rinishida saqlash uchun bo'sh ro'yxat ochamiz
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():  # Klasslar mosligi lug'atidagi har bir asl va yangi klass nomlarini aylanib chiqamiz
    class_path = os.path.join(DATASET_PATH, original_class)  # /content/drive/MyDrive/food101_dataset/hamburger  Har bir klassga tegishli rasmlar joylashgan papkaning to'liq manzilini hosil qilamiz
    print('class_path:', class_path)  # Konsolga tekshirilayotgan klass papkasi manzilini chop etamiz
    if not os.path.exists(class_path):  # Agar ko'rsatilgan klass papkasi diskda mavjud bo'lmasa
        print(f"Warning: {class_path} not found")  # Konsolga papka topilmagani haqida ogohlantirish chiqaramiz
        continue  # Siklning ushbu qadamini tashlab ketib, keyingi klassga o'tamiz
    for img in os.listdir(class_path):  # Klass papkasi ichidagi barcha fayllarni birma-bir aylanib chiqamiz
        if img.endswith(('.jpg', '.jpeg', '.png')):  # Fayl kengaytmasi rasm formatida (.jpg, .jpeg yoki .png) ekanligini tekshiramiz
            full_path = os.path.join(class_path, img)  # /content/drive/MyDrive/food101_dataset/hamburger/100057.jpg Rasm faylining to'liq manzilini (yo'lini) shakllantiramiz
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))  # (/content/drive/MyDrive/food101_dataset/hamburger/100057.jpg , 1)  Rasmning to'liq manzili va unga mos keladigan indeks raqamini juftlik qilib ro'yxatga qo'shamiz

np.random.shuffle(all_images)  # Ma'lumotlar bir xil turda ketma-ket kelib qolmasligi uchun barcha elementlarni tasodifiy tartibda aralashtiramiz
split = int(0.8 * len(all_images))  # Ma'lumotlarning 80 foizini o'quv (train) to'plami uchun ajratish nuqtasi (indeksini) hisoblaymiz
train_data = all_images[:split]  # Ro'yxatning boshidan to bo'linish nuqtasigacha bo'lgan qismini o'quv ma'lumotlari deb olamiz
val_data = all_images[split:]  # Bo'linish nuqtasidan oxirigacha bo'lgan qismini validatsiya (tekshirish) ma'lumotlari deb olamiz

train_images, train_labels = zip(*train_data)  # O'quv to'plamidagi rasmlar yo'li va teglarini alohida ikkita ro'yxatga ajratib (unzip) olamiz
val_images, val_labels = zip(*val_data)  # Validatsiya to'plamidagi rasmlar yo'li va teglarini alohida ikkita ro'yxatga ajratib (unzip) olamiz

print('all_images:', all_images)  # Konsolga barcha yig'ilgan rasmlar ro'yxatini chop etamiz (kodda biroz xira ko'ringan qism)

dataset = FoodDataset(train_images, train_labels)  # Ajratib olingan o'quv rasmlari va teglaridan foydalanib maxsus Dataset obyektini yaratamiz
print(len(dataset))  # Konsolga yaratilgan dataset ichidagi ma'lumotlarning umumiy sonini chop etamiz
img, lbl = dataset[0]  # Dataset-ning birinchi elementini (rasm va uning tegini) tekshirish uchun o'zgaruvchilarga yuklaymiz

In [ ]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform) #transform defolt None edi agar transform kiritsak parent classdagi if qismi ishga tushadi.Umumiy train uchun datasetni hosil qildik
val_dataset = FoodDataset(val_images, val_labels, transform=transform) # test uchun datasetni hosil qildik



In [ ]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle=True, num_workers=2) # batch_size bu 32 ta rasmni ai modelga train uchun beradigan qiymatimiz. shuffle-True -rasmlar ketma ketligini yodlab olmaslik sharti yani datasetdagi ketmaketlikni avval aralashtirib tashlab keyin train boladi. num_workers- backgroundda boshqa ishlarni amalga oshirishi mumkinligini aytyapmiz
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle=False, num_workers=2) # shuffle false test uchun ajratilgan datani ketma ketlikka asoslangan holda test qilamiz.Bunda rasmdagi ketma ketlik boyicha test amalga oshsa ham bolaveradi.random qilish shart emas

In [ ]:
# pretrainde model
model = mobilenet_v2(weights="IMAGENET1K_V1") # tayyor mobilenet_v2 modeldan foydalanyapmiz.weights="IMAGENET1K_V1" "Tasodifiy og'irliklar emas, balki oldindan o'qitilgan og'irliklarni yukla."
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # modelning barcha featurelarini ishlatamiz.1000 ta tayyor class bolsa 995 tani unut va biz kiritgan NUM_CLASSES ichidagi 5 ta classni train qil degan qismi

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # qurulmani aniqlaymiz yani train cpu yoki gpuda run bolayotganini aniqlaymiz
print('device', device)
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss() # loss function. model train bolayotganda barcha narsani ozaro taqqoslaydi. yani necha foiz buregerga yoki necha foiz kebabga oxshashini tahmin qiladi.
optimizer = optim.Adam(model.parameters(), lr=0.001) # train jarayonida parametrlarni har doim update qiladi (weight)
torch.backends.cudnn.benchmark = True # # Agar GPU (CUDA) ishlatilayotgan bo'lsa va kiruvchi tensorlar o'lchami bir xil bo'lsa,cuDNN eng tez convolution algoritmini avtomatik tanlaydi va keyingi hisoblashlarni tezlashtiradi.
print("criterion:",criterion )
print("optimizer:",optimizer )


In [ ]:
# ------------------
# Training Loop
# ------------------

NUM_EPOCHS = 10  # O'qitish davrlari (epoxalar) soni
best_accuracy = 0.0  # Eng yaxshi aniqlik ko'rsatkichini saqlash uchun o'zgaruvchi

for epoch in range(NUM_EPOCHS):  # Belgilangan epoxalar sonicha siklni aylantirish
    model.train()  # Modelni o'qitish (training) rejimiga o'tkazish
    running_loss = 0.0  # Joriy epoxadagi umumiy xatolikni hisoblash uchun o'zgaruvchi

    for images, labels in train_loader:  # Ma'lumotlar yuklagichidan (batch) rasmlar va teglarni olish
        # Rasmlar va teglarni hisoblash qurilmasiga (GPU yoki CPU) o'tkazish
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()  # Oldingi qadamdan qolgan gradientlarni (hosilalarni) nollash
        outputs = model(images)  # Rasmlarni modelga berib, bashorat (output) olish
        loss = criterion(outputs, labels)  # Bashorat va haqiqiy teglar o'rtasidagi xatolikni hisoblash
        loss.backward()  # Xatolikni orqaga tarqatish (gradientlarni hisoblash)
        optimizer.step()  # Model vaznlarini (vazn koeffitsiyentlarini) yangilash

        running_loss += loss.item()  # Joriy batch xatoligini umumiy xatolikka qo'shish

# ------------------
# Validation
# ------------------
model.eval()  # Modelni baholash/test rejimiga o'tkazish (Dropout va Batch Norm ni o'chiradi)
correct = 0   # To'g'ri topilgan jami namunalar sonini hisoblash o'zgaruvchisi
total = 0     # Jami tekshirilgan namunalar (rasmlar) soni

with torch.no_grad():  # Validatsiya jarayonida gradientlarni hisoblashni o'chirish (xotirani tejaydi)
    for images, labels in val_loader:  # Validatsiya ma'lumotlar yuklagichidan batchlarni olish
        # Rasmlar va teglarni tegishli qurilmaga (GPU yoki CPU) o'tkazish
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)  # Model orqali bashoratlarni olish
        _, predicted = torch.max(outputs, 1)  # Eng yuqori ehtimollikka ega bo'lgan klass indeksini aniqlash
        total += labels.size(0)  # Batchdagi umumiy elementlar sonini qo'shish
        correct += (predicted == labels).sum().item()  # To'g'ri topilgan bashoratlar sonini yig'ish

val_acc = 100 * correct / total  # Validatsiya aniqlik (accuracy) foizini hisoblash
# Epoxa natijalarini (o'rtacha xatolik va validatsiya aniqligini) ekranga chiqarish
print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.2f}%")

if val_acc > best_accuracy:  # Agar joriy aniqlik shu vaqtgacha bo'lgan eng yaxshi aniqlikdan yuqori bo'lsa
    best_accuracy = val_acc  # Eng yaxshi aniqlik qiymatini yangilash
    # Modelning eng yaxshi og'irlik (vazn) parametrlarini faylga saqlash
    torch.save(model.state_dict(), '/content/menu_detector.pth')
    print("Saved new best model!")  # Yangi eng yaxshi model saqlanganligi haqida xabar chiqarish

## menu_detector model usage

In [ ]:
# ------------------
# Required Imports
# ------------------
import torch  # PyTorch asosiy kutubxonasini yuklash
import torchvision.transforms as transforms  # Tasvirlarga ishlov berish funksiyalarini yuklash
from torchvision.models import mobilenet_v2  # MobileNetV2 arxitekturasini import qilish
from PIL import Image  # Tasvirlarni ochish va boshqarish uchun PIL kutubxonasi
from google.colab import files  # Google Colab-ga fayl yuklash funksiyasini yuklash
import io  # Bayt oqimlari (input/output) bilan ishlash kutubxonasi
import matplotlib.pyplot as plt  # Tasvirlarni grafik shaklda ekranga chiqarish kutubxonasi

# ------------------
# Define Semantic Classes
# ------------------
# Model tanishi kerak bo'lgan taomlar sinflari (klasslari) ro'yxati
CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']  # O'qitish tartibiga mos kelishi shart
NUM_CLASSES = len(CLASSES)  # Jami klasslar sonini aniqlash (bu yerda 5 ta)
# Har bir klass nomini uning indeksiga moslashtiruvchi lug'at yaratish ({'hamburger': 0, ...})
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}

# ------------------
# Transform for Uploaded Images (no augmentations!)
# ------------------
# Yangi yuklanadigan rasmlarni modelga moslab qayta ishlovchi transformatsiya zanjiri
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Rasm o'lchamini MobileNetV2 kutgan 224x224 o'lchamga keltirish
    transforms.ToTensor(),  # Rasmni PyTorch Tensor (matritsa) ko'rinishiga o'tkazish (0-255 dan 0.0-1.0 gacha)
    # ImageNet ma'lumotlar to'plamining standart o'rtacha (mean) va og'ish (std) qiymatlari bilan normallashtirish
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ------------------
# Load Model
# ------------------
model = mobilenet_v2(weights=None)  # Bo'sh (vaznlarsiz) MobileNetV2 model arxitekturasini yaratish
# Modelning oxirgi tasniflagich (classifier) qatlamini bizning klasslar soniga (5 taga) moslab o'zgartirish
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
# Saqlangan eng yaxshi og'irlik (vazn) parametrlarini CPU xotirasiga moslab yuklash
model.load_state_dict(torch.load('/content/menu_detector.pth', map_location='cpu'))

# Agar GPU (CUDA) mavjud bo'lsa undan foydalanish, aks holda CPU-ni tanlash
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)  # Amaldagi qurilma nomini ekranga chiqarish

model = model.to(device)  # Model arxitekturasini tanlangan qurilmaga (GPU yoki CPU) o'tkazish
model.eval()  # Modelni baholash/test rejimiga o'tkazish (Inference uchun tayyorlash)
# ------------------
# Upload & Predict
# ------------------
# Foydalanuvchiga rasm yuklash bo'yicha ko'rsatma matnini chiqarish
print("Upload one or more images of hamburger or hot dog:")
uploaded = files.upload()  # Google Colab-da kompyuterdan fayl yuklash oynasini ochish

for image_name in uploaded.keys():  # Yuklangan barcha rasmlarni ketma-ket siklda aylantirish
    # Yuklangan baytlar oqimidan rasmni ochish va uni RGB formatiga o'tkazish
    image = Image.open(io.BytesIO(uploaded[image_name])).convert('RGB')

    # # Display image
    plt.figure(figsize=(4, 4))  # Chiziladigan rasm oynasi o'lchamini belgilash (4x4 dyuym)
    plt.imshow(image)  # Rasmni grafik ekraniga joylashtirish
    plt.axis('off')  # Grafik o'qlarini (koordinata chiziqlarini) yashirish
    plt.title(f'Uploaded: {image_name}')  # Rasm tepasiga uning fayl nomini sarlavha qilib yozish
    plt.show()  # Tayyor bo'lgan tasvirni ekranda ko'rsatish

    # # Predict
    # Rasmni transformatsiya qilish, batch o'lchamini qo'shish (.unsqueeze(0)) va qurilmaga o'tkazish
    image_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():  # Bashorat qilishda gradientlar hisoblanishini o'chirish (tezlikni oshiradi)
        output = model(image_tensor)  # Rasmni modelga berib, logit (xom) natijalarni olish
        probs = torch.softmax(output, dim=1)[0]  # Logitlarni foizli ehtimollik ko'rinishiga (0-1) keltirish
        topk = torch.topk(probs, 4)  # Eng yuqori ehtimollikka ega bo'lgan top-4 ta klassni ajratib olish

    print("Prediction:")  # Ekran paneliga "Prediction:" sarlavhasini chiqarish
    for i in range(topk.indices.size(0)):  # Topilgan top-4 ta klass bo'yicha sikl aylantirish
        label = CLASSES[topk.indices[i]]  # Indeks bo'yicha tegishli klass nomini olish (masalan, 'hamburger')
        confidence = topk.values[i].item() * 100  # Klass ehtimolligini foiz ko'rinishiga o'tkazish
        print(f" - {label}: {confidence:.2f}%")  # Klass nomi va uning ishonchlilik foizini ekranga chiqarish